# IndicSpeak — Training Walkthrough

End-to-end: raw audio → SNAC tokens → compiled sequences → FSDP2 training → synthesis with your
checkpoint. Training is **token-in, token-out** — audio is SNAC-encoded once, offline:

```
audio + text ──> stage 1 tokenize ──> SNAC parquet ──> stage 2 compile ──> training parquet
                                                                                │
                              checkpoint  <── scripts/tts/train.sh (accelerate) <──┘
```

**Prerequisites**

- GPU node, environment installed (`./install.sh && source .venv/bin/activate`).
- An **extended tokenizer** with the frozen audio-token layout (`<|snac_0|>` = 128266,
  `len == 156942`). This repo does not build tokenizers — the canopylabs checkpoints ship one;
  point TOKENIZER below at a local copy of it.
  See [docs/tts/token_layout.md](../../docs/tts/token_layout.md).
- Audio as WAV/FLAC files plus transcripts. Any sample rate (resampled to 24 kHz mono).

This notebook drives a **tiny smoke run** so every stage finishes in minutes; the same configs
scale to real corpora by editing paths and step counts.

In [ ]:
import json
from pathlib import Path

import torch
import yaml

assert torch.cuda.is_available(), "training needs a GPU node"

# --- edit these ------------------------------------------------------------
TOKENIZER = "/path/to/checkpoints/llama-3-audio-tok_trimmed"
BASE_MODEL = "canopylabs/orpheus-3b-0.1-pretrained"  # or local orpheus_3B_pt
SNAC = "hubertsiuzdak/snac_24khz"  # or local snac_24khz dir
AUDIO_DIR = Path("/path/to/your/wavs")  # your audio files
# ----------------------------------------------------------------------------

WORK = Path("notebook_train").absolute()
for d in ("manifests", "tokenized", "compiled", "configs"):
    (WORK / d).mkdir(parents=True, exist_ok=True)
print("workspace:", WORK)

## 1. Build a JSONL manifest

Stage 1 reads one JSON object per line. Required: an audio path and a transcript; `language`,
`speaker`, `style` and `accent` are optional metadata that flow into the templates.

In [ ]:
# Toy example: pair each wav with a transcript. Replace with your real transcript source.
rows = [
    {
        "audio_filepath": str(wav),
        "text": wav.stem.replace("_", " "),  # <- put the real transcript here
        "language": "en",
        "speaker": "S1",
    }
    for wav in sorted(AUDIO_DIR.glob("*.wav"))[:16]  # 16 files is plenty for a smoke run
]
manifest = WORK / "manifests" / "train.jsonl"
manifest.write_text("".join(json.dumps(r) + "\n" for r in rows))
print(f"{len(rows)} rows -> {manifest}")
print(rows[0] if rows else "!! no wavs found — fix AUDIO_DIR")

## 2. Stage 1 — SNAC tokenize

GPU Ray actors encode audio to interleaved SNAC token ids (7 tokens/frame, offset from
`<|snac_0|>`), writing sharded Parquet plus a resume manifest — re-running skips completed
shards. Config reference: [docs/tts/configs.md](../../docs/tts/configs.md).

In [ ]:
tokenize_cfg = {
    "ray": {"address": None},
    "models": {"snac_model_path": SNAC, "tokenizer_path": TOKENIZER},
    "processing": {"workers_per_gpu": 1, "encode_batch_size": 16},
    "datasets": [
        {
            "source": {"jsonl": str(manifest)},
            "columns": {
                "audio": "audio_filepath",
                "text": "text",
                "language": "language",
                "speaker": "speaker",
            },
            "output": {"dir": str(WORK / "tokenized" / "train"), "rows_per_shard": 1000},
        }
    ],
}
cfg_path = WORK / "configs" / "tokenize.yaml"
cfg_path.write_text(yaml.safe_dump(tokenize_cfg, sort_keys=False))
print(cfg_path.read_text())

In [ ]:
!RAY_ADDRESS=local python -m bodhan_genai.tts.data.tokenize --config {cfg_path}

In [ ]:
import pyarrow.parquet as pq

tok_table = pq.read_table(sorted((WORK / "tokenized" / "train").glob("*.parquet"))[0])
print(tok_table.schema)
first = tok_table.to_pylist()[0]
print(f"text: {first['text']!r}")
print(
    f"{len(first['token_ids'])} snac tokens "
    f"(≈{len(first['token_ids']) / 7 * 0.085:.1f}s), first 7: {first['token_ids'][:7]}"
)
assert all(128266 <= t < 156938 for t in first["token_ids"]), "ids outside SNAC range!"

## 3. Stage 2 — compile training sequences

Applies the Llama chat template per row — basic TTS, or conversation (rows carrying
`is_conversation`) — and writes `input_ids` / `labels` / `length` Parquet, sorted
length-descending for packing. The Orpheus recipe trains with **full-sequence loss**:
`labels == input_ids`, no prompt masking.

In [ ]:
compile_cfg = {
    "models": {"tokenizer_path": TOKENIZER},
    "processing": {
        "training_mode": "sft",  # the only mode: SFT template, full loss
        "drop_style": False,
        "num_workers": 0,
        "random_seed": 42,
        "split_by_language": False,
        "split_by_source": False,
    },
    "input": {
        "datasets": [
            {
                "input_dir": str(WORK / "tokenized" / "train"),
                "output_dir": str(WORK / "compiled" / "train"),
                "source_name": "smoke",
            }
        ]
    },
}
cfg_path = WORK / "configs" / "compile.yaml"
cfg_path.write_text(yaml.safe_dump(compile_cfg, sort_keys=False))
!python -m bodhan_genai.tts.data.compile --config {cfg_path}

In [ ]:
comp_files = sorted((WORK / "compiled" / "train").rglob("*.parquet"))
comp = pq.read_table(comp_files[0]).to_pylist()
row = comp[0]
print(f"{len(comp)} sequences; first: {row['length']} tokens")
assert row["input_ids"] == row["labels"], "Orpheus route: labels must equal input_ids"
# structural sanity: sequence starts with <|start_of_human|> and contains <|start_of_speech|>
print("starts with:", row["input_ids"][:2], "| has start_of_speech:", 128257 in row["input_ids"])

## 4. Author the training config

The `training:` block goes straight into `transformers.TrainingArguments` (v5 names). Smoke-run
choices worth knowing:

- `max_seq_len` — packing builds fixed `[1, max_seq_len]` batches per GPU; OOM ⇒ lower it or
  keep activation checkpointing on (the shipped accelerate config does).
- `compile: false` — skips the 10–20 min first-step `torch.compile` warm-up. Turn it back on
  for real runs.
- `save_total_limit: null` is **required** — `BestAndLastCheckpointKeeper` owns retention
  (`checkpoint_retention:` block).
- Re-running with the same `output_dir` **auto-resumes** from the last checkpoint.

The shipped [configs/tts/train/pretrain.yaml](../../configs/tts/train/pretrain.yaml) /
[sft.yaml](../../configs/tts/train/sft.yaml) are the full-scale references.

In [ ]:
train_cfg = {
    "training_stage": "sft",
    "model": {
        "model_path": BASE_MODEL,
        "tokenizer_path": TOKENIZER,  # training resizes embeddings to match (156942)
        "max_seq_len": 4096,
        "attn_implementation": "flash_attention_2",
        "torch_dtype": "bfloat16",
        "activation_checkpointing": False,  # FSDP owns act-ckpt (accelerate config)
        "compile": False,  # smoke run: skip inductor warm-up
        "snac_model_path": SNAC,
    },
    "data": {
        "train": {
            "packing": {"backend": "auto", "rank_local": False, "equalize_rank_bins": True},
            "datasets": [{"path": str(WORK / "compiled" / "train"), "ratio": 1.0, "name": "smoke"}],
        },
    },
    "training": {
        "output_dir": str(WORK / "checkpoints" / "smoke"),
        "max_steps": 20,
        "gradient_accumulation_steps": 1,
        "learning_rate": 1.0e-4,
        "warmup_steps": 2,
        "lr_scheduler_type": "cosine",
        "optim": "adamw_torch_fused",
        "bf16": True,
        "logging_steps": 1,
        "save_strategy": "steps",
        "save_steps": 10,
        "eval_strategy": "no",
        "save_total_limit": None,
        "seed": 42,
        "dataloader_num_workers": 2,
        "remove_unused_columns": False,
    },
    "checkpoint_retention": {"last_n": 1, "best_k": 1, "metric": "eval_loss"},
    "logging": {"wandb_project": "bodhan-genai-smoke"},
}
cfg_path = WORK / "configs" / "train_smoke.yaml"
cfg_path.write_text(yaml.safe_dump(train_cfg, sort_keys=False))
print("wrote", cfg_path)

## 5. Launch

`scripts/tts/train.sh` wraps `accelerate launch` with the single-node FSDP2 config and the tuned
inductor/NCCL environment. `NUM_GPUS=N` overrides autodetection. **The cell blocks until
training finishes** — for real runs, launch from a terminal (or `tmux`) instead of a notebook
so the run survives kernel restarts.

In [ ]:
!cd {Path.cwd()} && NUM_GPUS=1 WANDB_MODE=disabled scripts/tts/train.sh {cfg_path}

## 6. Checkpoints, resume, monitoring

- Checkpoints land in `training.output_dir`; retention keeps the union of best-K (by
  `eval_loss`) and last-N.
- **Resume**: rerun the exact same command — `get_last_checkpoint` picks up where it stopped.
- **wandb**: `WANDB_MODE=offline scripts/tts/train.sh ...` on egress-less nodes, then
  `wandb sync <output_dir>/wandb/offline-run-*` later. `TrainingMetricsCallback` logs
  perplexity, MFU, peak GPU memory, and packing efficiency.

In [ ]:
ckpts = sorted((WORK / "checkpoints" / "smoke").glob("checkpoint-*"))
print("checkpoints:", [c.name for c in ckpts])
CKPT = str(ckpts[-1])

## 7. LoRA variant

Same config plus a `lora:` block (see [configs/tts/train/lora.yaml](../../configs/tts/train/lora.yaml)),
launched with `scripts/tts/train_lora.sh`. Produces compact adapter-only checkpoints; load them at
inference with `IndicTTSEngine(base_model, backend="hf", adapter_dir=...)`.

```yaml
lora:
  r: 32
  lora_alpha: 64
  lora_dropout: 0.05
  target_modules: [q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj]
```

## 8. Synthesize with your checkpoint

Pass the **extended tokenizer the model was trained with** — after training the embedding was
resized to 156942, so checkpoint and tokenizer agree.

In [ ]:
from IPython.display import Audio, display

from bodhan_genai.tts import IndicTTSEngine

with IndicTTSEngine(CKPT, tokenizer=TOKENIZER) as engine:
    r = engine.synthesize("The smoke run checkpoint says hello.", speaker="S1")
    display(Audio(r.audio, rate=r.sample_rate))

    # conversation mode works the same way once trained on conversation data:
    # engine.synthesize_conversation([
    #     {"speaker": "S1", "text": "How did the run go?"},
    #     {"speaker": "S2", "text": "Twenty steps, loss went down. Ship it."},
    # ])

## Scaling up

- Point `data.train.datasets` at your full compiled corpora with mixing `ratio`s, restore
  `compile: true`, raise `max_seq_len` (act-ckpt on), set epoch-based scheduling and
  `eval_strategy: steps` with a `val` split — [configs/tts/train/pretrain.yaml](../../configs/tts/train/pretrain.yaml)
  is the production-shaped reference.
- Serve the result: `CHECKPOINT=... scripts/tts/serve.sh` ([docs/tts/serving.md](../../docs/tts/serving.md)),
  or batch-synthesize a manifest with `scripts/tts/infer.sh`.
- Inference API tour: [notebooks/inference.ipynb](inference.ipynb).